# table_maintenance-nyc_taxi-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
spark.sql("CREATE TABLE IF NOT EXISTS lakehouse.silver.nyc_taxi_tm AS SELECT * FROM lakehouse.bronze.nyc_taxi_trips").show(truncate=False)
spark.sql("INSERT INTO lakehouse.silver.nyc_taxi_tm SELECT * FROM lakehouse.bronze.nyc_taxi_trips WHERE passenger_count > 3").show(truncate=False)
spark.sql("SELECT count(*) AS files FROM lakehouse.silver.nyc_taxi_tm.files").show(truncate=False)

## 4. Transform

In [ ]:
spark.sql("CALL lakehouse.system.rewrite_data_files(table => 'lakehouse.silver.nyc_taxi_tm', options => map('target-file-size-bytes','134217728'))").show(truncate=False)

## 5. Write

In [ ]:
spark.sql("CALL lakehouse.system.expire_snapshots(table => 'lakehouse.silver.nyc_taxi_tm', older_than => current_timestamp(), retain_last => 1)").show(truncate=False)
spark.sql("CALL lakehouse.system.remove_orphan_files(table => 'lakehouse.silver.nyc_taxi_tm')").show(truncate=False)

## 6. Verify

In [ ]:
spark.sql("SELECT count(*) AS snapshots FROM lakehouse.silver.nyc_taxi_tm.snapshots").show(truncate=False)
spark.sql("SELECT count(*) AS files FROM lakehouse.silver.nyc_taxi_tm.files").show(truncate=False)